# Docking Pose Consistency Tool - Demo Notebook

This notebook demonstrates how to use the Docking Pose Consistency Tool
to compare docked ligand poses against a reference pose.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from pathlib import Path

# Suppress warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

## 1. Setup and Load Data

Update these paths to point to your actual files.

In [ ]:
PROTEIN_PATH = '../examples/protein.pdb'
REFERENCE_PATH = '../examples/reference_pose.sdf'
POSES_PATH = '../examples/docked_poses/'
OUTPUT_DIR = Path('../example_output')
OUTPUT_DIR.mkdir(exist_ok=True)

from src.utils import setup_logging
logger = setup_logging(OUTPUT_DIR, verbose=False)

from src.io_handlers import load_protein, load_reference, load_molecules

protein = load_protein(PROTEIN_PATH)
ref = load_reference(REFERENCE_PATH)
queries = load_molecules(POSES_PATH)

print(f'Reference: {ref.name if ref else "FAILED"}')
print(f'Loaded {len(queries)} docked poses')

## 2. RMSD Analysis (Method 1 & 2)

In [ ]:
from src.alignment import batch_rmsd_analysis

# Compute RMSD against reference (poses kept as-docked, no alignment)
rmsd_results = batch_rmsd_analysis(ref, queries)
rmsd_df = pd.DataFrame(rmsd_results)
rmsd_df[['compound', 'heavy_atom_rmsd', 'mcs_rmsd', 'mcs_num_atoms', 'mcs_fraction_ref']]

## 3. Interaction Fingerprint Analysis (Method 3)

In [ ]:
from src.interactions import batch_ifp_analysis

ifp_results, fp_matrix = batch_ifp_analysis(PROTEIN_PATH, ref, queries)
ifp_df = pd.DataFrame(ifp_results)
ifp_df[['compound', 'ifp_tanimoto', 'ifp_dice', 'ifp_overlap_pct', 'n_hbonds', 'n_hydrophobic']]

## 4. Pharmacophore Analysis (Method 4)

In [ ]:
from src.pharmacophore import batch_pharmacophore_analysis

pharma_results = batch_pharmacophore_analysis(ref, queries)
pharma_df = pd.DataFrame(pharma_results)
pharma_df[['compound', 'pharmacophore_score', 'feature_overlap_pct', 'matched_features']]

## 5. Orientation Analysis (Method 5)

In [ ]:
from src.orientation import batch_orientation_analysis

orient_results = batch_orientation_analysis(ref, queries)
orient_df = pd.DataFrame(orient_results)
orient_df[['compound', 'com_distance', 'principal_axis_angle', 'is_flipped', 'orientation_score']]

## 6. Shape Similarity (Method 6)

In [ ]:
from src.shape import batch_shape_analysis

shape_results = batch_shape_analysis(ref, queries)
shape_df = pd.DataFrame(shape_results)
shape_df[['compound', 'usr_similarity', 'usrcat_similarity', 'gaussian_overlap', 'shape_tanimoto']]

## 7. Classification

In [ ]:
from src.classification import PoseClassifier

# Merge all metrics
merged = {}
for dataset in [rmsd_results, ifp_results, pharma_results, orient_results, shape_results]:
    for r in dataset:
        name = r.get('compound')
        if name not in merged:
            merged[name] = {}
        merged[name].update(r)

all_metrics = list(merged.values())

classifier = PoseClassifier()
classifications = classifier.classify_batch(all_metrics)

for m, c in zip(all_metrics, classifications):
    m['classification'] = c['classification']
    m['consensus_score'] = c['consensus_score']

cls_df = pd.DataFrame(classifications)
cls_df[['compound', 'classification', 'consensus_score']]

## 8. Clustering & Visualization

In [ ]:
from src.clustering import build_feature_matrix, hierarchical_clustering, compute_pca
import matplotlib.pyplot as plt

feature_matrix, feat_names, feat_cols = build_feature_matrix(all_metrics)

if len(all_metrics) >= 3:
    cluster_result = hierarchical_clustering(feature_matrix, feat_names)
    pca_result = compute_pca(feature_matrix, feat_names)
    
    # PCA plot
    coords = pca_result['coordinates']
    class_colors = {
        'CONSERVED BINDING MODE': '#2ecc71',
        'PARTIAL CONSERVATION': '#f39c12',
        'DIFFERENT POSE': '#e74c3c',
        'FLIPPED ORIENTATION': '#9b59b6',
        'OUTLIER': '#95a5a6',
    }
    colors = [class_colors.get(m.get('classification', ''), '#333') for m in all_metrics]
    
    plt.figure(figsize=(10, 8))
    plt.scatter(coords[:, 0], coords[:, 1], c=colors, s=100, edgecolors='black')
    for i, name in enumerate(feat_names):
        plt.annotate(name[:12], (coords[i, 0], coords[i, 1]), fontsize=7)
    plt.xlabel(f'PC1 ({pca_result["explained_variance"][0]*100:.1f}%)')
    plt.ylabel(f'PC2 ({pca_result["explained_variance"][1]*100:.1f}%)')
    plt.title('Pose Similarity - PCA Projection')
    plt.tight_layout()
    plt.show()

## 9. Interactive 3D Viewer

In [ ]:
try:
    import py3Dmol
    from rdkit import Chem
    
    view = py3Dmol.view(width=800, height=500)
    
    # Reference in green
    ref_block = Chem.MolToMolBlock(ref.mol)
    view.addModel(ref_block, 'sdf')
    view.setStyle({'model': 0}, {'stick': {'color': '#2ecc71', 'radius': 0.15}})
    
    # Queries in blue
    for i, q in enumerate(queries[:15]):
        if q.mol:
            q_block = Chem.MolToMolBlock(q.mol)
            view.addModel(q_block, 'sdf')
            view.setStyle({'model': i+1}, {'stick': {'color': '#3498db', 'radius': 0.1, 'opacity': 0.5}})
    
    view.zoomTo()
    view.show()
except ImportError:
    print('py3Dmol not installed. Use: pip install py3Dmol')

## 10. Export Results

In [ ]:
results_df = pd.DataFrame(all_metrics)
results_df.to_csv(OUTPUT_DIR / 'results.csv', index=False, float_format='%.3f')
print(f'Results saved to {OUTPUT_DIR / "results.csv"}')
results_df[['compound', 'heavy_atom_rmsd', 'ifp_tanimoto', 'pharmacophore_score',
             'shape_tanimoto', 'consensus_score', 'classification']]